In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from typing import List, Tuple, Optional, Union
from dataclasses import dataclass
from tqdm import tqdm
import logging
from datetime import date, timedelta
from torchinfo import summary
import os
import math
import statsmodels.api as sm
import random

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("RetPredictor")

def set_seed(seed: int = 32) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

class WeightedMSELoss(nn.Module):
    def forward(self, input, target, weight):
        return (weight * (input - target) ** 2).mean()

class PriceReturnDataset(torch.utils.data.Dataset):
    def __init__(self, 
                 seq_len: int,
                 n_pred: int,
                 df: pd.DataFrame, 
                 features: List[str], 
                 targets: np.ndarray,
                 prob_col: Optional[str]=None,
                 n_regimes: int=2,
                 is_test: bool=False,
                ) -> None:
        self.seq_len = seq_len
        self.n_pred = n_pred
        self.df = df
        self.targets = targets
        self.features = features
        self.prob_col = prob_col
        self.n_regimes = n_regimes
        self.is_test = is_test
        ndays = df.shape[0]
        assert ndays == targets.shape[0], "Mismatch in df shape and targets shape"

    def __len__(self) -> int:
        if self.is_test:
            return (self.targets.shape[0] - self.seq_len + 1) // self.n_pred
        return self.targets.shape[0] - self.seq_len - self.n_pred + 1

    def __getitem__(self, idx: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        if self.is_test:
            begin = idx*self.n_pred
            end = idx*self.n_pred +self.seq_len-1
        else:
            begin = idx
            end = idx+self.seq_len-1
        data_array = self.df.loc[begin:end, self.features].values # pd includes both start and begin
        ar_pred = self.df.loc[end:end+self.n_pred-1, "arPred"].values
        # (seq_len, nfeatures)
        if self.prob_col is None:
            cols = [f"p{i}" for i in range(self.n_regimes)]
            prob_array = self.df.loc[end:end+self.n_pred-1, cols].values
        else:
            prob_array = self.df.loc[end:end+self.n_pred-1, self.prob_col].values
        return (
            torch.tensor(data_array, dtype=torch.float32),
            self.targets[end:end+self.n_pred],
            torch.tensor(prob_array, dtype=torch.float32),
            torch.tensor(ar_pred, dtype=torch.float32),
        )
    
    
class PositionalEncoding(nn.Module):
    def __init__(self, seq_len: int, d_model: int) -> None:
        super().__init__()
        pe = torch.zeros(seq_len, d_model)
        pos = torch.arange(0, seq_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        self.pe = pe.unsqueeze(0)  # (1, seq_len, d_model)

    def forward(self, x: torch.tensor) -> torch.tensor:
        return x + self.pe[:, 0:x.size(1), :].to(x.device)


class EncoderPredictor(nn.Module):
    def __init__(self, 
                 n_features: int,
                 seq_len: int,
                 ret_index: int,
                 d_model: int=64, 
                 n_head: int=4, 
                 n_layers: int=2, 
                 dropout: float=0.1,
                 n_pred: int=10, #10,
                ) -> None:
        super().__init__()
        self.ret_index = ret_index

        self.relu = nn.ReLU()
        
        # Project features → model dimension
        self.input_projection = nn.Linear(n_features, d_model)

        # Positional encoding
        self.pos_encoding = PositionalEncoding(seq_len=seq_len, d_model=d_model)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_head,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Output head
        self.fc1 = nn.Sequential(
            nn.Linear(d_model, 32),
            self.relu,
            nn.Linear(32, 1),
        )

        self.fc2 = nn.Sequential(
            nn.Linear(seq_len, 32),
            self.relu,
            nn.Linear(32, n_pred),
        )

        self.n_autoreg_returns = min(3, seq_len)
        self.fc_final = nn.Linear(self.n_autoreg_returns + n_pred, n_pred)
        
        # print a summary of the model
        logger.info(self.summary(seq_len, n_features))

    def forward(self, x_input: torch.tensor) -> torch.tensor:
        # x: (B, T, F) where T: seq_len, F: num features
        x = self.input_projection(x_input)

        # Add positional encoding
        x = self.pos_encoding(x)

        # Transformer
        x = self.transformer(x)

        # predict output
        x = self.fc1(x)

        x = torch.squeeze(x, 2)
        x = self.fc2(x)

        x_final = torch.cat((x_input[:, x_input.shape[1]-self.n_autoreg_returns:, self.ret_index], 
                             x), dim=1)
        x_out = self.fc_final(x_final)

        return x_out
    
    def summary(self, seq_len: int, n_features: int) -> None:
        model_stats = summary(self, input_size=(1, seq_len, n_features))
        logger.info(f"Total trainable parameters: {model_stats.trainable_params}")
    
@dataclass
class ModelConfig:
    use_gpu: bool = True
    src_seq_len: int = 21
    d_model: int = 64 #32 #96 #128 #64
    n_head: int = 8
    n_layer: int = 12
    n_pred: int = 40
    n_regimes: int = 2
    p_dropout: float = 0.1
    learning_rate: float = 0.002
    weight_decay: float = 0.01
    betas: Tuple[float, float] = (0.9, 0.999)
    n_epochs: int = 15
    batch_size: int = 32
    shuffle: bool = False
    max_grad_norm: float = 1.0
    save_model: bool = False
    features: Tuple[str] = ("retPeriod",
                           "priceConvexity",
                           "volatility1Month", 
                           "longShortMA",
                           )
    ret_index: int = 0 # index of retPeriod in features
    float_dtype = torch.float32
    use_sampling: bool = False
        
    # lags used in AR model
    n_lags: int = 3
        
    # trade strat paramss
    train_begin_dt: date = date(2000, 2, 4)
    train_end_dt: date = date(2017, 3, 1)
        
    # for data
    input_dir: str = "/kaggle/input/datasets/samitahlawat/stockeoddata/stocks"
    spy_input_dir: str = "/kaggle/input/datasets/samitahlawat/stockeoddata"
    output_dir: str = "/kaggle/working"
        
class ModelContainer:
    def __init__(self, config: ModelConfig) -> None:
        # device
        # Define the device
        self.config = config
        device = "cuda" if torch.cuda.is_available() and config.use_gpu else "cpu"
        logger.info(f"Using device: {device}")
        
        self.device = torch.device(device)
        
        self.model = EncoderPredictor(n_features=len(config.features),
                                      seq_len=config.src_seq_len,
                                      ret_index=config.ret_index,
                                      d_model=config.d_model,
                                      n_head=config.n_head, 
                                      n_layers=config.n_layer,
                                      dropout=config.p_dropout,
                                      n_pred=config.n_pred,
                                     ).to(self.device)
        self.loss_fn = WeightedMSELoss().to(self.device).to(config.float_dtype)
        self.optimizer = optim.AdamW(self.model.parameters(), 
                                     lr=config.learning_rate,
                                     betas=config.betas, 
                                     weight_decay=config.weight_decay)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer,
                                                              mode='min',
                                                              factor=0.1, 
                                                              patience=3,
                                                             )

    def finetune_one_epoch_using_data(self, x: torch.tensor, y: torch.tensor, weight: torch.tensor) -> float:
        self.model.train()
        self.optimizer.zero_grad()
        encoder_input = x.to(self.device).to(self.config.float_dtype) # (b, seq, nfeatures)
        tgt = y.to(self.device).to(self.config.float_dtype)
        wgt = weight.to(self.device).to(self.config.float_dtype)
        pred = self.model(encoder_input)

        # Compute the loss
        loss = self.loss_fn(pred, tgt, wgt)

        # Backpropagate the loss
        loss.backward()

        # Update the weights
        self.optimizer.step()
        
        total_loss = loss.item()
        self.model.eval()
        return total_loss
    
    def train_one_epoch(self, epoch: int, dataloader: torch.utils.data.DataLoader) -> float:
        batch_iterator = tqdm(dataloader, desc=f"Processing Epoch {epoch:02d}")
        total_loss = 0.0
        for batch in batch_iterator:
            self.optimizer.zero_grad()
            encoder_input = batch[0].to(self.device).to(self.config.float_dtype) # (b, seq, nfeatures)
            tgt = batch[1].to(self.device).to(self.config.float_dtype)
            prob = batch[2].to(self.device).to(self.config.float_dtype)
            pred = self.model(encoder_input)

            # Compute the loss
            loss = self.loss_fn(pred, tgt, prob)

            # Backpropagate the loss
            loss.backward()

            # Update the weights
            self.optimizer.step()
            
            total_loss += loss.item()
        return total_loss / len(dataloader.dataset)
        
    def train(self, dataset: PriceReturnDataset) -> None:
        dataloader = torch.utils.data.DataLoader(
            dataset=dataset,
            batch_size=self.config.batch_size,
            shuffle=self.config.shuffle,
            drop_last=False,
        )
        loss_arr = []
        lr_arr = []
        self.model.train()
        
        for epoch in range(self.config.n_epochs):
            loss = self.train_one_epoch(epoch, dataloader)
            
            self.scheduler.step(loss)
        
            logger.info(
                f"Epoch [{epoch+1}/{self.config.n_epochs}] "
                f"Loss: {loss:.4f} "
                f"LR = {self.optimizer.param_groups[0]['lr']}"
            )
            loss_arr.append(loss)
            lr_arr.append(self.optimizer.param_groups[0]['lr'])
            
        # set to eval mode after training
        self.model.eval()
        logger.info(loss_arr)
        logger.info(lr_arr)

    def save(self, name: str) -> None:
        model_filename = os.path.join(self.config.output_dir, f"{name}.pth")
        torch.save({
            'model_state_dict': self.model.state_dict(),
        }, model_filename)
        
    def load(self, name: str) -> None:
        model_filename = os.path.join(self.config.output_dir, f"{name}.pth")
        checkpoint = torch.load(model_filename, map_location=self.device)
        self.model.load_state_dict(checkpoint)
        self.model.eval()
        
    def predict(self, x: torch.tensor) -> torch.tensor:
        x = x.to(self.device).to(self.config.float_dtype)
        return self.model(x)

    
class FeatureCalculator:
    @staticmethod
    def adjustPrices(df: pd.DataFrame) -> pd.DataFrame:
        df.loc[:, "Adj Close"] = df.loc[:, "Close"]
        return df

    @staticmethod
    def calculateARModelPred(df: pd.DataFrame,
                             ycol: str,
                             xcol: str,
                             n_lags: int,
                             begin: int,
                             end: int,
                             n_pred: int,
                        ) -> pd.DataFrame:
        y = df.loc[begin:end-1, ycol].values
        x = df.loc[:, xcol].values
        X = []
        for i in range(n_lags):
            xi = x[begin-i:end-i]
            X.append(xi[:, np.newaxis])
        Xarr = np.concatenate(X, axis=1)
        Xarr = sm.add_constant(Xarr, has_constant="add")
        ar_model = sm.OLS(y, Xarr)
        ar_model = ar_model.fit()
        Xarr = np.ones((1, n_lags+1), dtype=np.float32)
        
        for i in range(begin, df.shape[0], n_lags):
            Xarr[0, 0] = 1.0
            for j in range(n_lags):
                Xarr[0, 1+j] =  x[i-j]
            ylast2 = x[i+1]
            for k in range(n_lags):
                ypred = ar_model.predict(Xarr)[0]
                df.loc[i+k, "arPred"] = ypred
                for j in range(n_lags-1):
                    Xarr[0, j+2] = Xarr[0, j+1]
                Xarr[0, 1] = ylast2
                ylast2 = ypred
        
        return df

    @staticmethod
    def calculateRegime(df: pd.DataFrame,
                        y: str,
                        X: Tuple[str],
                        begin: int = 21,
                        end: int = -2,
                        n_regimes: int = 2,
                        train_end_date: Optional[date] = None,
                  ) -> pd.DataFrame:
        if end < 0:
            end = df.shape[0] + end
        
        train_end = end
        if train_end_date is not None:
            train_end = df.loc[df.Date.dt.date >= train_end_date, :].index[0]

        vol = df.loc[begin:train_end, "volatility1Month"].values
        max_val = vol.max()
        min_val = vol.min()
        p0 = (max_val - df.loc[begin:end, "volatility1Month"].values) / (max_val - min_val)
        p0 = np.clip(p0, 0, 1)
        p1 = 1.0 - p0
        regime = np.where(p1 >= 0.5, 1, 0)
        df.loc[begin:end, "regime"] = regime
        df.loc[begin:end, "p0"] = p0
        df.loc[begin:end, "p1"] = p1
        return df
    
    @staticmethod
    def calculateReturn(df: pd.DataFrame, colname: str, period: int) -> pd.DataFrame:
        price = df.loc[:, "Adj Close"].values
        ret = price[period:] / price[0:-period] - 1
        df.loc[0:period, colname] = 0.0
        df.loc[period+1:, colname] = ret[0:-1]
        last_ret_data = pd.DataFrame({"Date": [df.loc[df.shape[0]-1, "Date"] + timedelta(days=1)], colname: [ret[-1]]})
        df = pd.concat([df, last_ret_data], ignore_index=True, sort=False)
        return df
    
    @staticmethod
    def priceMA(dfData: pd.DataFrame, lookback: int) -> pd.DataFrame:
        closepx = dfData.loc[:dfData.shape[0]-2, "Adj Close"].values
        dfData.loc[:, "longShortMA"] = 0.0
        for i in range(3*lookback, dfData.shape[0]-2, 1):
            longMA = np.mean(closepx[i-3*lookback:i])
            shortMA = np.mean(closepx[i-1*lookback:i])
            dfData.loc[i, "longShortMA"] = longMA/shortMA - 1.0
        return dfData
    
    @staticmethod
    def priceConvexity(dfData: pd.DataFrame, lookback: int) -> pd.DataFrame:
        closepx = dfData.loc[:dfData.shape[0]-2, "Adj Close"].values
        convexity = (-closepx[(3*lookback-1):] + 2*closepx[(2*lookback-1):-lookback] - closepx[0:-(3*lookback-1)]) / closepx[0:-(3*lookback-1)]
        dfData.loc[3*lookback:, "priceConvexity"] = convexity
        return dfData
        
    @staticmethod
    def volatility(dfData: pd.DataFrame) -> pd.DataFrame:
        lookback = 22
        ret1vals = dfData.loc[:, "retPeriod"].values
        volat = np.zeros(dfData.shape[0], dtype=np.float64)
        for i in range(lookback-1, volat.shape[0], 1):
            volat[i] = np.std(ret1vals[i-lookback+1:i+1])
        dfData.loc[:,"volatility1Month"] = volat
        
        
        return dfData
    
    @staticmethod
    def calculate(dfData: pd.DataFrame, 
                  period: int, 
                  begin_date: Optional[date],
                  end_date: date, # training end date
                  n_regimes: int,
                  n_lags: int,
                  n_pred: int,
                 ) -> pd.DataFrame:
        colname = "retPeriod"
        dfData = FeatureCalculator.adjustPrices(dfData)
        dfData = FeatureCalculator.calculateReturn(dfData, colname=colname, period=period)
        dfData = FeatureCalculator.volatility(dfData)
        dfData = FeatureCalculator.priceConvexity(dfData, lookback=period)
        dfData = FeatureCalculator.priceMA(dfData, lookback=period)
        dfData = FeatureCalculator.calculateRegime(dfData, 
                                                   y=colname, 
                                                   X=(), 
                                                   n_regimes=n_regimes,
                                                   train_end_date=end_date,
                                                  )
        
        for i in range(n_regimes):
            avg_i = dfData.loc[dfData.regime == i, colname].mean()
            logger.info(f"regime {i} avg: {avg_i}")
        avg = dfData.loc[:, colname].mean()
        logger.info(f"overall avg: {avg}")
        price = dfData.loc[:dfData.shape[0]-2, "Adj Close"].values
        dfData.loc[:, "nextPeriodRet"] = 0.0
        dfData.loc[0:dfData.shape[0]-2-period, "nextPeriodRet"] = price[period:]/price[0:-period] - 1
        end = dfData.loc[dfData.Date.dt.date >= end_date, :].index[0]
        dfData = FeatureCalculator.calculateARModelPred(dfData,
                                                        ycol="nextPeriodRet",
                                                        xcol="retPeriod",
                                                        n_lags=n_lags,
                                                        begin=21,
                                                        end=end,
                                                        n_pred=n_pred,
                                                       )
        float_cols = dfData.select_dtypes(include='float').columns.tolist()
        dfData.loc[:, float_cols] = dfData.loc[:, float_cols].fillna(0.0)
        if begin_date is not None:
            dfData = dfData.loc[dfData.Date.dt.date >= begin_date, :].reset_index(drop=True)
        return dfData
    
    
class Predictor:
    def __init__(self, config: ModelConfig, security: str, load_cached: bool = False) -> None:
        # 1. process data
        self.selected_security = security
        self.config = config
        self.df_dict = {}
        
        new_securities = []
        df_sz = None
        first_date = None
        last_date = None
        if security == "SPY":
            df = pd.read_csv(os.path.join(self.config.spy_input_dir, f"{security}.csv"), parse_dates=["Date"])
        else:
            df = pd.read_csv(os.path.join(self.config.input_dir, f"{security}.csv"), parse_dates=["Date"])
        self.df = FeatureCalculator.calculate(df,
                                              period=config.n_pred, 
                                              begin_date=config.train_begin_dt,
                                              end_date=config.train_end_dt,
                                              n_regimes=config.n_regimes,
                                              n_lags=config.n_lags,
                                              n_pred=config.n_pred,
                                             )
        # 2. create dataset for training (for each regime)
        self.train_datasets = [None] * config.n_regimes
        self.model_containers = [None] * config.n_regimes
        for i in range(config.n_regimes):
            self.train_datasets[i] = self.create_training_dataset(i)
            self.model_containers[i] = ModelContainer(config)
        
        if load_cached:
            self.model_container.load()

        
    def create_training_dataset(self, iregime: int) -> PriceReturnDataset:
        df = self.df.loc[self.df.Date.dt.date <= self.config.train_end_dt, :].reset_index(drop=True)
            
        targets = df.loc[:, "nextPeriodRet"].values
        prob_col = f"p{iregime}"
        
        dset = PriceReturnDataset(seq_len=self.config.src_seq_len,
                                  n_pred=self.config.n_pred,
                                  df=df,
                                  features=self.config.features, 
                                  targets=targets,
                                  prob_col=prob_col,
                                  n_regimes=self.config.n_regimes,
                                 )
        return dset
    
    def create_testing_dataset(self) -> PriceReturnDataset:
        #df = self.df.loc[self.df.Date.dt.date > self.config.train_end_dt, :].reset_index(drop=True)
        df = self.df
            
        targets = df.loc[:, "nextPeriodRet"].values
        dset = PriceReturnDataset(seq_len=self.config.src_seq_len,
                                  n_pred=self.config.n_pred,
                                  df=df,
                                  features=self.config.features, 
                                  targets=targets,
                                  n_regimes=self.config.n_regimes,
                                  is_test=True,
                                 )
        return dset
    
    def train_model(self) -> None:
        for i in range(self.config.n_regimes):
            self.model_containers[i].train(self.train_datasets[i])
        
    def predict(self) -> None:
        dset = self.create_testing_dataset()
        dataloader = torch.utils.data.DataLoader(
            dataset=dset,
            batch_size=1,
            shuffle=False,
            drop_last=False,
        )
        self.df.loc[:, "predictedRet"] = 0.0
        begin = 0
        test_set_begin = self.df.loc[self.df.Date.dt.date > self.config.train_end_dt, :].index[0]
        for x, y, prob, ar_pred in dataloader:
            x = x.to(self.model_containers[0].device).to(self.config.float_dtype) # (b, n_sec, seq, nfeatures)
            y = y.to(self.model_containers[0].device).to(self.config.float_dtype)
            prob = prob.to(self.model_containers[0].device).to(self.config.float_dtype)
            pred = 0
            if self.config.use_sampling:
                selected = torch.multinomial(prob[0, :, :], num_samples=1).cpu().detach().numpy()[0, 0]
                predi = self.model_containers[selected].predict(x)
                pred = prob[:, :, selected] * predi
            else:
                for i in range(self.config.n_regimes):
                    predi = self.model_containers[i].predict(x)
                    pred += prob[:, :, i] * predi
            pred = pred.cpu().detach().numpy()
            ar_pred = ar_pred.cpu().detach().numpy()
            
            ind_begin = begin+self.config.src_seq_len-1
            if ind_begin + pred.shape[1]-1 < self.df.shape[0]:
                #self.df.loc[ind_begin:ind_begin + pred.shape[1]-1, "arPred"] = ar_pred[0, :]
                self.df.loc[ind_begin:ind_begin + pred.shape[1]-1, "predictedRet"] = pred[0, :]
            begin = begin + pred.shape[1]

            # finetune using new data (already used for prediction)
            if begin > test_set_begin:
                for i in range(self.config.n_regimes):
                    self.model_containers[i].finetune_one_epoch_using_data(x, y, prob[:, :, i])
            
        self.df.to_csv(os.path.join(self.config.output_dir, f"{self.selected_security}_pred.csv"))
    

In [ ]:
config = ModelConfig()
securities = ['SPY']
for sec in securities:
    logger.info(f"##########{sec}############")
    predictor = Predictor(config, sec)
    predictor.train_model()
    predictor.predict()